# Sign → Text — Model Comparison & Thesis Metrics

Trains **three architectures** on one identical split and produces every metric a
supervisor typically asks for:

- Accuracy & loss curves (train vs validation) per model
- Confusion matrix (counts **and** row-normalized) per model
- Per-class precision / recall / F1 + macro & weighted averages
- Top-1 / Top-3 accuracy, Cohen's κ, Matthews CC, macro ROC-AUC
- Parameters, model size, training time, inference latency
- Cross-model comparison table (CSV) + overlaid curves + bar charts

Architectures: **(A) CNN-BiGRU** (your current hybrid), **(B) BiLSTM + Attention**
(recurrent), **(C) Transformer encoder** (self-attention). Reuses cached `X.npy`/`y.npy`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Tip: Runtime > Change runtime type > GPU  (much faster for the Transformer/LSTM)

In [ ]:
import os, json, time, numpy as np, tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                             cohen_kappa_score, matthews_corrcoef, roc_auc_score, top_k_accuracy_score)
import pandas as pd

OUT_DIR     = '/content/drive/MyDrive/HAND_MODEL'
METRICS_DIR = os.path.join(OUT_DIR, 'thesis_metrics')
os.makedirs(METRICS_DIR, exist_ok=True)

AUG_PER_SAMPLE = 12
TEST_SPLIT = 0.15
VAL_SPLIT  = 0.15      # fraction of the remaining (train+val) used for validation
EPOCHS     = 120
BATCH_SIZE = 32
SEED       = 42
np.random.seed(SEED); tf.random.set_seed(SEED)
plt.rcParams['figure.dpi'] = 130

## Load cached features + labels

In [ ]:
X = np.load(os.path.join(OUT_DIR, 'X.npy'))
y = np.load(os.path.join(OUT_DIR, 'y.npy'))

lm_path = os.path.join(OUT_DIR, 'label_map.json')
if os.path.exists(lm_path):
    label_map = json.load(open(lm_path))
else:
    raise FileNotFoundError('label_map.json not found in OUT_DIR — save it next to X.npy/y.npy')

labels = [k for k, _ in sorted(label_map.items(), key=lambda kv: kv[1])]
N = len(labels)
assert len(np.unique(y)) == N, f'class count mismatch: y has {len(np.unique(y))}, label_map has {N}'
print('X', X.shape, '| y', y.shape, '|', N, 'classes')
print(labels)

## Augmentation (training set only) — same ops as the training notebook

In [ ]:
def _slot_present(pts): return np.abs(pts).sum(axis=(2,3)) > 0
def aug_rotate(x, max_deg=15.0):
    th = np.deg2rad(np.random.uniform(-max_deg, max_deg)); c,s = np.cos(th), np.sin(th)
    p = x.reshape(-1,2,21,3).copy(); xx,yy = p[...,0].copy(), p[...,1].copy()
    p[...,0]=xx*c-yy*s; p[...,1]=xx*s+yy*c; return p.reshape(x.shape)
def aug_scale(x, lo=0.9, hi=1.1): return (x*np.random.uniform(lo,hi)).astype(np.float32)
def aug_jitter(x, sigma=0.02):
    p = x.reshape(-1,2,21,3).copy(); m=_slot_present(p)[...,None,None]
    return (p+np.random.normal(0,sigma,p.shape)*m).reshape(x.shape).astype(np.float32)
def aug_mirror(x):
    p = x.reshape(-1,2,21,3).copy(); p[...,0]*=-1.0; p=p[:,[1,0],:,:]
    return p.reshape(x.shape).astype(np.float32)
def aug_timewarp(x):
    T=x.shape[0]; a=int(np.random.uniform(0,0.2*T)); b=int(np.random.uniform(0.8*T,T-1))
    if b<=a: a,b=0,T-1
    idx=np.clip(np.round(np.linspace(a,b,T)),0,T-1).astype(int); return x[idx].astype(np.float32)
def augment(x):
    if np.random.rand()<0.5: x=aug_timewarp(x)
    if np.random.rand()<0.7: x=aug_rotate(x)
    if np.random.rand()<0.5: x=aug_scale(x)
    if np.random.rand()<0.5: x=aug_mirror(x)
    if np.random.rand()<0.7: x=aug_jitter(x)
    return x.astype(np.float32)

## One fair split: train / val / test (test never augmented)

In [ ]:
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=TEST_SPLIT, stratify=y, random_state=SEED)
X_tr, X_val, y_tr, y_val = train_test_split(X_tmp, y_tmp, test_size=VAL_SPLIT, stratify=y_tmp, random_state=SEED)

aug_X, aug_y = [], []
for xi, yi in zip(X_tr, y_tr):
    aug_X.append(xi); aug_y.append(yi)
    for _ in range(AUG_PER_SAMPLE): aug_X.append(augment(xi)); aug_y.append(yi)
X_tr_a = np.array(aug_X, np.float32); y_tr_a = np.array(aug_y)
perm = np.random.permutation(len(X_tr_a)); X_tr_a, y_tr_a = X_tr_a[perm], y_tr_a[perm]
print('train(aug):', X_tr_a.shape, '| val:', X_val.shape, '| test:', X_test.shape)

from sklearn.utils.class_weight import compute_class_weight
cw = compute_class_weight('balanced', classes=np.arange(N), y=y_tr)
class_weight = {i: float(w) for i, w in enumerate(cw)}
T, F = X.shape[1], X.shape[2]

## Three architectures

In [ ]:
from tensorflow.keras import layers, models

def build_cnn_bigru():
    inp = layers.Input((T, F))
    x = layers.Conv1D(64,3,padding='same',activation='relu')(inp); x = layers.BatchNormalization()(x)
    x = layers.Conv1D(128,3,padding='same',activation='relu')(x); x = layers.BatchNormalization()(x)
    x = layers.Bidirectional(layers.GRU(64,return_sequences=True,dropout=0.2))(x)
    x = layers.Bidirectional(layers.GRU(64,dropout=0.2))(x)
    x = layers.Dense(64,activation='relu')(x); x = layers.Dropout(0.4)(x)
    out = layers.Dense(N,activation='softmax')(x)
    return models.Model(inp,out,name='CNN_BiGRU')

def build_bilstm_attention():
    inp = layers.Input((T, F))
    x = layers.Bidirectional(layers.LSTM(128,return_sequences=True,dropout=0.2))(inp)
    x = layers.Bidirectional(layers.LSTM(64,return_sequences=True,dropout=0.2))(x)
    score = layers.Dense(1)(x)                 # additive attention pooling over time
    score = layers.Softmax(axis=1)(score)
    ctx = layers.Multiply()([x, score])
    ctx = layers.Lambda(lambda z: tf.reduce_sum(z, axis=1))(ctx)
    d = layers.Dense(64,activation='relu')(ctx); d = layers.Dropout(0.4)(d)
    out = layers.Dense(N,activation='softmax')(d)
    return models.Model(inp,out,name='BiLSTM_Attention')

class PositionalEmbedding(layers.Layer):
    def __init__(self, T, d, **kw):
        super().__init__(**kw); self.T=T; self.pos=layers.Embedding(T,d)
    def call(self, x):
        return x + self.pos(tf.range(self.T))

def build_transformer(d_model=128, heads=4, ff=256, blocks=2, drop=0.2):
    inp = layers.Input((T, F))
    x = layers.Dense(d_model)(inp)
    x = PositionalEmbedding(T, d_model)(x)
    for _ in range(blocks):
        a = layers.MultiHeadAttention(num_heads=heads, key_dim=d_model//heads, dropout=drop)(x, x)
        x = layers.LayerNormalization(epsilon=1e-6)(x + a)
        f = layers.Dense(ff, activation='relu')(x); f = layers.Dense(d_model)(f); f = layers.Dropout(drop)(f)
        x = layers.LayerNormalization(epsilon=1e-6)(x + f)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64,activation='relu')(x); x = layers.Dropout(0.4)(x)
    out = layers.Dense(N,activation='softmax')(x)
    return models.Model(inp,out,name='Transformer')

BUILDERS = {'CNN_BiGRU': build_cnn_bigru, 'BiLSTM_Attention': build_bilstm_attention, 'Transformer': build_transformer}

## Metric & plot helpers (figures saved to Drive)

In [ ]:
def save_history_plot(hist, name):
    h = hist.history
    fig, ax = plt.subplots(1,2, figsize=(11,4))
    ax[0].plot(h['accuracy'], label='train'); ax[0].plot(h['val_accuracy'], label='val')
    ax[0].set_title(f'{name} — Accuracy'); ax[0].set_xlabel('epoch'); ax[0].set_ylabel('accuracy'); ax[0].legend(); ax[0].grid(alpha=.3)
    ax[1].plot(h['loss'], label='train'); ax[1].plot(h['val_loss'], label='val')
    ax[1].set_title(f'{name} — Loss'); ax[1].set_xlabel('epoch'); ax[1].set_ylabel('loss'); ax[1].legend(); ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.savefig(os.path.join(METRICS_DIR, f'{name}_curves.png'), bbox_inches='tight'); plt.show()

def save_confusion(y_true, y_pred, name, normalize=False):
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(N)).astype(float)
    if normalize:
        cm = cm / np.clip(cm.sum(1, keepdims=True), 1, None)
    fig, ax = plt.subplots(figsize=(min(1+0.5*N,14), min(1+0.5*N,14)))
    im = ax.imshow(cm, cmap='Blues'); fig.colorbar(im, fraction=0.046)
    ax.set_xticks(range(N)); ax.set_xticklabels(labels, rotation=90, fontsize=8)
    ax.set_yticks(range(N)); ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ttl = f'{name} — Confusion' + (' (normalized)' if normalize else ' (counts)')
    ax.set_title(ttl)
    thr = cm.max()/2 if cm.max()>0 else .5
    for i in range(N):
        for j in range(N):
            v = cm[i,j]
            if v>0:
                txt = f'{v:.2f}' if normalize else f'{int(v)}'
                ax.text(j,i,txt,ha='center',va='center',fontsize=7, color='white' if v>thr else 'black')
    plt.tight_layout()
    suffix = 'cm_norm' if normalize else 'cm'
    plt.savefig(os.path.join(METRICS_DIR, f'{name}_{suffix}.png'), bbox_inches='tight'); plt.show()

def save_perclass_f1(report_dict, name):
    f1 = [report_dict[l]['f1-score'] for l in labels]
    plt.figure(figsize=(min(1+0.5*N,14),4))
    plt.bar(range(N), f1); plt.xticks(range(N), labels, rotation=90, fontsize=8)
    plt.ylim(0,1); plt.ylabel('F1'); plt.title(f'{name} — per-class F1'); plt.grid(axis='y', alpha=.3)
    plt.tight_layout(); plt.savefig(os.path.join(METRICS_DIR, f'{name}_f1.png'), bbox_inches='tight'); plt.show()

## Train + evaluate all three

In [ ]:
results = {}
histories = {}
y_test_oh = np.eye(N)[y_test]

for name, builder in BUILDERS.items():
    print('\n' + '='*60 + f'\nTraining {name}\n' + '='*60)
    tf.keras.backend.clear_session(); np.random.seed(SEED); tf.random.set_seed(SEED)
    model = builder()
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    cbs = [tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=15, restore_best_weights=True),
           tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=6, min_lr=1e-5)]
    t0 = time.time()
    hist = model.fit(X_tr_a, y_tr_a, validation_data=(X_val,y_val), epochs=EPOCHS,
                     batch_size=BATCH_SIZE, class_weight=class_weight, callbacks=cbs, verbose=2)
    train_time = time.time()-t0
    histories[name] = hist

    # ---- evaluate on the held-out TEST set ----
    probs = model.predict(X_test, verbose=0)
    y_pred = probs.argmax(1)
    _ = model.predict(X_test[:1], verbose=0)  # warm up
    ti = time.time(); _ = model.predict(X_test, verbose=0); infer_ms = (time.time()-ti)/len(X_test)*1000

    rep = classification_report(y_test, y_pred, target_names=labels, output_dict=True, zero_division=0)
    pd.DataFrame(rep).transpose().to_csv(os.path.join(METRICS_DIR, f'{name}_report.csv'))
    try: auc = roc_auc_score(y_test_oh, probs, multi_class='ovr', average='macro')
    except Exception: auc = float('nan')
    mfile = os.path.join(METRICS_DIR, f'{name}.keras'); model.save(mfile)

    results[name] = {
        'params': int(model.count_params()),
        'size_MB': round(os.path.getsize(mfile)/1e6, 2),
        'train_time_s': round(train_time,1),
        'infer_ms_per_sample': round(infer_ms,2),
        'test_accuracy': round(accuracy_score(y_test,y_pred),4),
        'top3_accuracy': round(top_k_accuracy_score(y_test, probs, k=3, labels=np.arange(N)),4),
        'macro_F1': round(rep['macro avg']['f1-score'],4),
        'weighted_F1': round(rep['weighted avg']['f1-score'],4),
        'cohen_kappa': round(cohen_kappa_score(y_test,y_pred),4),
        'mcc': round(matthews_corrcoef(y_test,y_pred),4),
        'macro_AUC': round(float(auc),4),
        'best_val_acc': round(max(hist.history['val_accuracy']),4),
    }
    print(json.dumps(results[name], indent=2))
    save_history_plot(hist, name)
    save_confusion(y_test, y_pred, name, normalize=False)
    save_confusion(y_test, y_pred, name, normalize=True)
    save_perclass_f1(rep, name)

## Cross-model comparison (table + overlaid curves + bar charts)

In [ ]:
summary = pd.DataFrame(results).transpose()
summary.to_csv(os.path.join(METRICS_DIR, 'comparison_summary.csv'))
json.dump(results, open(os.path.join(METRICS_DIR, 'comparison_summary.json'),'w'), indent=2)
print(summary.to_string())

# overlaid validation accuracy / loss
fig, ax = plt.subplots(1,2, figsize=(12,4))
for name, h in histories.items():
    ax[0].plot(h.history['val_accuracy'], label=name)
    ax[1].plot(h.history['val_loss'], label=name)
ax[0].set_title('Validation accuracy'); ax[0].set_xlabel('epoch'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].set_title('Validation loss'); ax[1].set_xlabel('epoch'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig(os.path.join(METRICS_DIR,'compare_curves.png'), bbox_inches='tight'); plt.show()

# bar chart: test accuracy & macro-F1
names = list(results.keys())
acc = [results[n]['test_accuracy'] for n in names]; f1 = [results[n]['macro_F1'] for n in names]
xx = np.arange(len(names)); w=0.38
plt.figure(figsize=(7,4))
plt.bar(xx-w/2, acc, w, label='Test accuracy'); plt.bar(xx+w/2, f1, w, label='Macro F1')
plt.xticks(xx, names, rotation=15); plt.ylim(0,1); plt.grid(axis='y', alpha=.3); plt.legend()
for i,(a,b) in enumerate(zip(acc,f1)):
    plt.text(i-w/2, a+.01, f'{a:.2f}', ha='center', fontsize=8); plt.text(i+w/2, b+.01, f'{b:.2f}', ha='center', fontsize=8)
plt.title('Model comparison (held-out test)')
plt.tight_layout(); plt.savefig(os.path.join(METRICS_DIR,'compare_bars.png'), bbox_inches='tight'); plt.show()

print('\nAll figures, CSVs, JSON and the 3 models saved to:', METRICS_DIR)

## What got saved (for the thesis)
In `HAND_MODEL/thesis_metrics/`:
- `<model>_curves.png` — accuracy & loss vs epoch
- `<model>_cm.png` / `<model>_cm_norm.png` — confusion matrices
- `<model>_f1.png` — per-class F1
- `<model>_report.csv` — precision/recall/F1/support per class
- `compare_curves.png`, `compare_bars.png` — cross-model
- `comparison_summary.csv` / `.json` — the master table
- `<model>.keras` — each trained model

To deploy any of these in the app, copy its `.keras` to the project root as `model.keras`
(its labels are the same `label_map.json` you trained with). The app preprocessing is
model-agnostic, so all three are drop-in compatible.